In [ ]:
import os
from kaggle_secrets import UserSecretsClient

# 1) Add your PAT as a Kaggle secret: Settings → Secrets → "MY_GITHUB_TOKEN"
#    (must have repo + workflow scope so the pipeline can push artifacts)
os.environ["GITHUB_TOKEN"] = UserSecretsClient().get_secret("MY_GITHUB_TOKEN")

# 2) Fresh clone into the Kaggle working dir (NOT /content — that's Colab)
!rm -rf sports_prediction_model
!git clone -q https://github.com/andrewkemmer/sports_prediction_model.git sports_prediction_model

# 3) Install every dependency the pipeline imports (backend, not the Streamlit frontend)
!pip install -q shap \
    lightgbm xgboost scikit-learn pandas numpy scipy joblib requests openpyxl pyarrow tqdm
print("setup done")

In [ ]:
import os

# --- OPTIONAL overrides (omit everything for a normal daily run) ---

# Custom window: only set these for one-off backfills, NOT daily runs
os.environ["NHL_START_DATE"] = "2024-01-01"
os.environ["NHL_END_DATE"]   = "2026-09-24"   # omit = runs through today

# Full repull: only for a rebuild; omit for daily (cache handles incremental)
os.environ["NHL_FULL_REPULL"] = "1"   # set once, then remove

print("run options set")

In [ ]:
import os
import subprocess

repo = "/kaggle/working/sports_prediction_model"
cmd = ["python", "nhl-backend/backend/master_pipeline.py"]

result = subprocess.run(cmd, cwd=repo, env=os.environ.copy(), capture_output=False)

# The pipeline's delivery phase already pushes nhl-backend/data_delivery/ to
# GitHub itself. Fail loudly if it didn't complete — Kaggle marks the run failed.
if result.returncode != 0:
    raise SystemExit(f"Pipeline failed with exit code {result.returncode}")
print("Pipeline completed — artifacts pushed to GitHub by the delivery sync.")

In [ ]:
import os
import subprocess

repo = "/kaggle/working/sports_prediction_model"

# Confirm main moved: fetch and compare HEAD to the pre-run HEAD
check = subprocess.run(
    ["git", "log", "--oneline", "-1"],
    cwd=repo, capture_output=True, text=True,
)
print("Repo HEAD after run:", check.stdout.strip())

# Sanity: newest dated artifact on main
ls = subprocess.run(
    ["git", "ls-tree", "-r", "--name-only", "origin/main"],
    cwd=repo, capture_output=True, text=True,
)
datelated = [f for f in ls.stdout.splitlines() if "nhl-backend/data_delivery/" in f and "_2026" in f]
print("Latest dated NHL artifacts:", sorted(datelated)[-3:] if datelated else "none found")